In [2]:
from pathlib import Path
import cv2
import numpy as np
import os


# =========================
# SETTINGS
# =========================

VIDEO_EXTENSIONS = (".mp4",)

KEEP_THREE_CHANNELS = True      # Recommended for DeepLabCut
SKIP_ALREADY_GRAYSCALE = True   # Avoid unnecessary recompression
FRAMES_TO_CHECK = 10            # Number of frames used to decide if video is already grayscale
GRAY_TOLERANCE = 2              # Allows tiny compression-related channel differences


# =========================
# HELPER FUNCTION:
# Check whether video is already grayscale
# =========================

def is_video_grayscale(video_path, frames_to_check=10, tolerance=2):
    """
    Returns True if sampled frames are visually grayscale.

    A video may still be stored/read as 3 channels, but if B, G, and R are
    equal or nearly equal, it is visually grayscale.
    """

    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        frame_indices = range(frames_to_check)
    else:
        step = max(total_frames // frames_to_check, 1)
        frame_indices = range(0, total_frames, step)

    checked = 0

    for frame_idx in frame_indices:
        if checked >= frames_to_check:
            break

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()

        if not ret:
            continue

        # If OpenCV somehow reads a true 1-channel frame
        if len(frame.shape) == 2:
            checked += 1
            continue

        if len(frame.shape) == 3 and frame.shape[2] == 3:
            b, g, r = cv2.split(frame)

            bg_diff = np.max(np.abs(b.astype(np.int16) - g.astype(np.int16)))
            gr_diff = np.max(np.abs(g.astype(np.int16) - r.astype(np.int16)))

            if bg_diff > tolerance or gr_diff > tolerance:
                cap.release()
                return False

            checked += 1
        else:
            cap.release()
            return False

    cap.release()

    if checked == 0:
        raise RuntimeError(f"Could not read any frames from video: {video_path}")

    return True


# =========================
# HELPER FUNCTION:
# Convert one video in place
# =========================

def convert_video_to_grayscale_inplace(video_path, keep_three_channels=True):
    """
    Converts one .mp4 video to grayscale and overwrites the original file.

    The output keeps:
    - the same filename
    - the same folder
    - the .mp4 format

    If keep_three_channels=True, the video looks grayscale but frames are saved
    as 3-channel BGR, which is safest for DeepLabCut.
    """

    video_path = Path(video_path)

    if video_path.suffix.lower() != ".mp4":
        raise ValueError(f"Only .mp4 videos are supported by this pipeline: {video_path}")

    temp_path = video_path.with_name(video_path.stem + "_GRAYSCALE.mp4")

    if temp_path.exists():
        temp_path.unlink()

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if fps <= 0:
        cap.release()
        raise RuntimeError(f"Could not read FPS from video: {video_path}")

    if width <= 0 or height <= 0:
        cap.release()
        raise RuntimeError(f"Could not read video dimensions: {video_path}")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        str(temp_path),
        fourcc,
        fps,
        (width, height),
        isColor=keep_three_channels,
    )

    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create temporary video: {temp_path}")

    frames_written = 0

    try:
        while True:
            ret, frame = cap.read()

            if not ret:
                break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            if keep_three_channels:
                gray = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

            writer.write(gray)
            frames_written += 1

    finally:
        cap.release()
        writer.release()

    if frames_written == 0:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(f"No frames were written for video: {video_path}")

    # Only replace the original after successful conversion
    os.replace(temp_path, video_path)

    print(f"Converted and overwritten: {video_path}")


# =========================
# MAIN PIPELINE:
# Convert all .mp4 videos in ROOT_DIR and subfolders
# =========================

def convert_folder_to_grayscale_inplace(
    root_dir,
    extensions=(".mp4",),
    skip_already_grayscale=True,
    keep_three_channels=True,
):
    root_dir = Path(root_dir)

    videos = [
        p for p in root_dir.rglob("*")
        if p.suffix.lower() in extensions
        and "_TEMP_GRAYSCALE" not in p.stem
    ]

    print(f"Found {len(videos)} .mp4 videos.")

    converted = 0
    skipped = 0
    failed = 0

    for video_path in videos:
        print("\n----------------------------------------")
        print(f"Checking: {video_path}")

        try:
            if skip_already_grayscale:
                if is_video_grayscale(
                    video_path,
                    frames_to_check=FRAMES_TO_CHECK,
                    tolerance=GRAY_TOLERANCE,
                ):
                    print("Already grayscale — skipping.")
                    skipped += 1
                    continue

            convert_video_to_grayscale_inplace(
                video_path,
                keep_three_channels=keep_three_channels,
            )
            converted += 1

        except Exception as e:
            print(f"FAILED: {video_path}")
            print(f"Reason: {e}")
            failed += 1

    print("\n========================================")
    print("Finished.")
    print(f"Converted: {converted}")
    print(f"Skipped:   {skipped}")
    print(f"Failed:    {failed}")
    print("========================================")



In [4]:
ROOT_DIR = Path(r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\Stefanos_RIdataset\20171027_35males")

convert_folder_to_grayscale_inplace(
    root_dir=ROOT_DIR,
    extensions=VIDEO_EXTENSIONS,
    skip_already_grayscale=SKIP_ALREADY_GRAYSCALE,
    keep_three_channels=KEEP_THREE_CHANNELS,
)

Found 105 .mp4 videos.

----------------------------------------
Checking: C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\Stefanos_RIdataset\20171027_35males\DAY1_20171027\20171027_V1_cropped_rois\mouse9041_Day01.mp4
Converted and overwritten: C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\Stefanos_RIdataset\20171027_35males\DAY1_20171027\20171027_V1_cropped_rois\mouse9041_Day01.mp4

----------------------------------------
Checking: C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\Stefanos_RIdataset\20171027_35males\DAY1_20171027\20171027_V1_cropped_rois\mouse9042_Day01.mp4
Converted and overwritten: C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\Stefanos_RIdataset\20171027_35males\DAY1_20171027\20171027_V1_cropped_rois\mouse9042_Day01.mp4

----------------------------------------
Checking: C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\Stefanos_RIdataset\20171027_35ma